# Mirror Download Server - Colab Runner

This notebook acts as the runner for the Telegram Mirror Bot. It handles dependency installation, Google Drive mounting, and starting the server with a public tunnel.

### Instructions:
1.  **Mount Drive**: Run the first cell to mount Google Drive.
2.  **Setup**: Run the setup cell to install dependencies (uses pre-built libtorrent wheel from PyPI).
3.  **Config**: Ensure your `config.py` and `credentials.json` are in your Drive (referenced in setup).
4.  **Run**: Start the server.

In [2]:
# @title 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive Mounted")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive Mounted


In [3]:
# @title 2. Install Dependencies
import os
import sys
import glob

# --- Step A: Remove broken/incompatible system libtorrent ---
print(f"🧹 Removing incompatible system libtorrent...")
!apt-get remove -y -qq python3-libtorrent 2>/dev/null

# Remove any stale libtorrent .so files from site-packages
for f in glob.glob("/usr/local/lib/python3.*/dist-packages/libtorrent*"):
    if os.path.isfile(f):
        os.remove(f)
        print(f"  Removed stale: {f}")
    elif os.path.isdir(f):
        import shutil
        shutil.rmtree(f)
        print(f"  Removed stale dir: {f}")

# Also remove from system python paths
for f in glob.glob("/usr/lib/python3/dist-packages/libtorrent*"):
    if os.path.isfile(f):
        os.remove(f)
        print(f"  Removed stale: {f}")

# --- Step B: Install system packages ---
print(f"📦 Installing system packages...")
!apt-get update -qq
!apt-get install -y -qq ffmpeg

# --- Step C: Install libtorrent from PyPI (pre-built wheel for Python 3.12) ---
print(f"📦 Installing libtorrent from PyPI (pre-built wheel)...")
!pip install --force-reinstall --no-cache-dir libtorrent==2.0.11

# Verify libtorrent
try:
    # Clear any cached import failures
    if "libtorrent" in sys.modules:
        del sys.modules["libtorrent"]
    import libtorrent as lt
    print(f"✅ Libtorrent installed successfully: {lt.version}")
except Exception as e:
    print(f"❌ Libtorrent installation failed: {e}")
    print("   Torrent features will be disabled, but the server will still work.")

# --- Step D: Install Python dependencies ---
print(f"📦 Installing Python dependencies...")
!pip install -q python-dotenv pyngrok nest_asyncio uvicorn fastapi python-multipart

# --- Step E: Install project requirements ---
PROJECT_DIR = "/content/drive/MyDrive/mirror_downloader"

if not os.path.exists(PROJECT_DIR):
    print(f"⚠️ {PROJECT_DIR} not found. Please upload your project files to this location in Drive.")
else:
    os.chdir(PROJECT_DIR)
    print(f"📂 Changed directory to {PROJECT_DIR}")
    if os.path.exists("requirements.txt"):
        !pip install -q -r requirements.txt
    else:
        print(f"⚠️ requirements.txt not found")
    print(f"✅ All Dependencies Installed!")

🧹 Removing incompatible system libtorrent...
  Removed stale dir: /usr/local/lib/python3.12/dist-packages/libtorrent-2.0.11.dist-info
  Removed stale dir: /usr/local/lib/python3.12/dist-packages/libtorrent
📦 Installing system packages...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
📦 Installing libtorrent from PyPI (pre-built wheel)...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 89.5 MB/s eta 0:00:00a 0:00:01


✅ Libtorrent installed successfully: 2.0.11.0
📦 Installing Python dependencies...
📂 Changed directory to /content/drive/MyDrive/mirror_downloader
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 737.3/737.3 kB 13.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 445.5/445.5 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.9/394.9 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.1/87.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.2/214.2 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.3 MB/s eta 0:00:00
✅ All Dependencies Installed!


In [4]:
# @title 3. Configuration Setup
# Ensure we have necessary config files.
# You can edit this cell to set environment variables directly if you don't use .env file

import os

# Set standard env vars for Colab
os.environ['IS_COLAB'] = 'true'
os.environ['TEMP_DOWNLOAD_DIR'] = '/content/downloads'

# Make sure download dir exists
os.makedirs('/content/downloads', exist_ok=True)

print("✅ Configuration Loaded")

✅ Configuration Loaded


In [6]:
# @title 4. Start Server & Tunnel
import nest_asyncio
import uvicorn
import threading
import sys
import os
import shutil

# Ensure we are in the correct directory
PROJECT_DIR = "/content/drive/MyDrive/mirror_downloader"
if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)
    if PROJECT_DIR not in sys.path:
        sys.path.append(PROJECT_DIR)
    print(f"📂 Working directory set to: {os.getcwd()}")
else:
    print(f"⚠️ Warning: {PROJECT_DIR} not found. Running in current directory: {os.getcwd()}")

# Load environment variables explicitly
from dotenv import load_dotenv

# Check for .env file
if not os.path.exists('.env'):
    if os.path.exists('.env.example'):
        print("⚠️ .env not found, creating from .env.example...")
        shutil.copy('.env.example', '.env')
        print("✅ Created .env from example. PLEASE EDIT THIS FILE IN YOUR DRIVE WITH YOUR TOKENS!")
        # Load defaults so it doesn't crash immediately, but user needs to edit
        load_dotenv(override=True)
    else:
        print("❌ .env and .env.example not found! Please upload them.")
else:
    load_dotenv(override=True)
    print("✅ Loaded .env file")


# Check libtorrent
try:
    import libtorrent as lt
    print(f"✅ Libtorrent imported successfully: {lt.version}")
except ImportError as e:
    print(f"❌ Failed to import libtorrent: {e}")
    print("   Please run Step 2 to build libtorrent from source.")
except Exception as e:
    print(f"❌ Unexpected error importing libtorrent: {e}")

from config import get_settings

# Apply nest_asyncio to allow nested event loops in Colab
nest_asyncio.apply()

settings = get_settings()

# Authenticate ngrok
NGROK_TOKEN = os.getenv("NGROK_AUTHTOKEN", "")

if not NGROK_TOKEN:
    print("⚠️ NGROK_AUTHTOKEN not found. Please set it in .env or config.py")
else:
    # Ensure it's set for subprocesses
    os.environ["NGROK_AUTHTOKEN"] = NGROK_TOKEN
    print(f"✅ Ngrok Token Set: {NGROK_TOKEN[:4]}...{NGROK_TOKEN[-4:]}")

# Run the server
print("Starting FastAPI server...")
# We use python -m uvicorn to ensure it runs in the same environment
!python -m uvicorn main:app --host 0.0.0.0 --port 8000

📂 Working directory set to: /content/drive/MyDrive/mirror_downloader
✅ Loaded .env file
✅ Libtorrent imported successfully: 2.0.11.0
✅ Ngrok Token Set: 2wdI...RpuU
Starting FastAPI server...
INFO:     Started server process [6621]
INFO:     Waiting for application startup.
2026-02-19 03:24:40,096 - main - INFO - Starting Mirror Download Server
2026-02-19 03:24:40,104 - task_manager - INFO - Loaded 23 tasks from persistence
2026-02-19 03:24:40,107 - main - INFO - No files to clean up from cancelled/failed tasks

🔐 Google Drive Authentication
✅ Using existing valid token.json
2026-02-19 03:24:40,399 - services.telegram_service - INFO - Telegram service initialized
2026-02-19 03:24:40,399 - main - INFO - Telegram bot: ENABLED
2026-02-19 03:24:40,435 - services.torrent_service - INFO - Libtorrent high-performance settings applied
2026-02-19 03:24:40,435 - services.torrent_service - INFO - Libtorrent session initialized
2026-02-19 03:24:40,492 - pyngrok.process - INFO - Updating authtoken f

In [ ]:
# @title 5. Keep Alive (Optional)
# Run this cell to prevent Colab from disconnecting due to idleness
import time
while True:
    time.sleep(60)
    print(".", end="", flush=True)